# Spatial maps of Lumina models (Ay 119 final project)

Learing spatial differences between cosmological models in Lumina simulations.

JWST has found more bright galaxies at z ≳ 10 than pre-launch ΛCDM models predicted, and several very different explanations reproduce the same UV luminosity function: brightening every galaxy (enhanced star-formation
efficiency), adding scatter to the M_UV–halo relation (bursty star formation), or changing the cosmology itself (early dark energy, which lets massive halos form earlier). Because these models match the same galaxy counts, we try to break such degeneracies through their clustering and cosmic variance. The goal is to tackle the model separation problem that upcoming wide surveys like Roman partly designed to do with clustering measurements.

This notebook builds directly on `compute_UVLF_cv.py`, which is work that I've done during the past few weeks for my SURF project, which computes cosmic variance and the UVLF from raw data in Lumina and Lumina EDE. The central question is can a CNN distinguish Lumina Lambda CDM and Lumina EDE data runs using their spatial galaxy maps at z=10?

Some experimental/model design and why:
- A1 uses the existing `+delta_MUV=1` model, every galaxy brightened by one magnitude.
- C1 uses the existing `+sigma_UV=1` model, one magnitude of scatter in M_UV at fixed halo. This is a standin for stocastic explanations.
- EDE uses the original Lumina EDE UV magnitudes.
- The field dimensions come from the existing 4.4 by 2.2 arcmin and `9.2<z<10.9`, which is an approximation for the JWST NIRCam survey geometry (12.4 × 6.2 × 369 cMpc).
- One map combines 4 by 4 of the original fields. In the data, one original field contains only about 2 to 4 galaxies at `MUV<-19.5`. So this means a large image of a single field would be basically zero. Hence I combined 4 by 4 neighboring fields into one map. Each map is still a real spatial region, but now contains enough galaxies for a CNN.
    - Map size: approximately 49.5 by 24.7 by 369 cMpc.
    - Image size: 32 by 16 pixels.
    - Maps per model: 200.
- All three samples are matched to the same total number of galaxies, so we prevent the CNN to just count instead of learning patterns.
- Every pixel represents the number of galaxies at that spatial location.

In [ ]:
#Import
from pathlib import Path
import copy
import h5py
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,TensorDataset
#constants and helpers copied from my SURF script compute_UVLF_cv.py
from lumina_geometry import (
    fuv_to_MUV,
    calculate_rectangle_field_lengths,
    lumina_cosmology,
    z_center,
    z_min,
    z_max,
    nircam_width,
    nircam_height,
    MUV_limits,
    best_delta_MUV,
    best_sigma_UV,
    random_seed
)

np.random.seed(7)
torch.manual_seed(7)
device="cpu"
print("device:",device)

In [ ]:
#processing the data because the raw data is like 11GB. This only extract the necessary info
data_dir=Path("data") #put lumina_z10.hdf5 and luminaEDE_z10.hdf5 here (not included in the repo)
project_dir=Path(".")
lcdm_file=data_dir/"lumina_z10.hdf5"
ede_file=data_dir/"luminaEDE_z10.hdf5"
cache_file=project_dir/"cache/uv_samples_all_limits.npz"
build_again=False

def build_uv_cache(lcdm_file,ede_file,cache_file,chunk_size=1000000):
    field_lengths=calculate_rectangle_field_lengths(cosmology=lumina_cosmology,z_center=z_center,z_min=z_min,z_max=z_max,width_angle=nircam_width,height_angle=nircam_height)
    with h5py.File(lcdm_file,"r") as file:
        box_size=float(file.attrs["boxsize"])/1000
    n_side=np.floor(box_size/field_lengths).astype(int)
    covered_lengths=n_side*field_lengths
    rng=np.random.default_rng(random_seed)

    # everything brighter than -19.5 is a subset of MUV<-19.5.
    cache_limit=np.max(MUV_limits)
    a1_positions=[]
    a1_muv=[]
    c1_positions=[]
    c1_muv=[]
    ede_positions=[]
    ede_muv=[]

    #selecting logic below is from my compute UVLF scripts as well
    # Lambda CDM
    with h5py.File(lcdm_file,"r") as file:
        number_of_galaxies=len(file["fuv"])
        for start in range(0,number_of_galaxies,chunk_size):
            stop=min(start+chunk_size,number_of_galaxies)
            positions_raw=file["pos"][start:stop]
            fuv=file["fuv"][start:stop]
            valid=(fuv>0)&np.isfinite(fuv)
            valid=valid&np.all(np.isfinite(positions_raw),axis=1)
            positions=positions_raw[valid]/1000
            muv=fuv_to_MUV(fuv[valid])
            inside=np.all(positions<covered_lengths,axis=1)
            positions=positions[inside]
            muv=muv[inside]

            muv_a1=muv-best_delta_MUV
            muv_c1=muv+rng.normal(0,best_sigma_UV,size=len(muv))
            bright_a1=muv_a1<cache_limit
            bright_c1=muv_c1<cache_limit
            a1_positions.append(positions[bright_a1])
            a1_muv.append(muv_a1[bright_a1])
            c1_positions.append(positions[bright_c1])
            c1_muv.append(muv_c1[bright_c1])

    # EDE
    with h5py.File(ede_file,"r") as file:
        number_of_galaxies=len(file["fuv"])
        for start in range(0,number_of_galaxies,chunk_size):
            stop=min(start+chunk_size,number_of_galaxies)

            positions_raw=file["pos"][start:stop]
            fuv=file["fuv"][start:stop]

            valid=(fuv>0)&np.isfinite(fuv)
            valid=valid&np.all(np.isfinite(positions_raw),axis=1)

            positions=positions_raw[valid]/1000
            muv=fuv_to_MUV(fuv[valid])

            inside=np.all(positions<covered_lengths,axis=1)
            positions=positions[inside]
            muv=muv[inside]

            bright=muv<cache_limit
            ede_positions.append(positions[bright])
            ede_muv.append(muv[bright])

    a1_positions=np.concatenate(a1_positions).astype(np.float32)
    a1_muv=np.concatenate(a1_muv).astype(np.float32)
    c1_positions=np.concatenate(c1_positions).astype(np.float32)
    c1_muv=np.concatenate(c1_muv).astype(np.float32)
    ede_positions=np.concatenate(ede_positions).astype(np.float32)
    ede_muv=np.concatenate(ede_muv).astype(np.float32)

    cache_file=Path(cache_file)
    cache_file.parent.mkdir(parents=True,exist_ok=True)

    np.savez_compressed(cache_file,a1_positions=a1_positions,a1_muv=a1_muv,c1_positions=c1_positions,c1_muv=c1_muv,ede_positions=ede_positions,ede_muv=ede_muv,field_lengths=field_lengths,box_size=box_size,muv_limits=MUV_limits)

    print("saved cache:",cache_file)
    print("A1:",len(a1_positions))
    print("C1:",len(c1_positions))
    print("EDE:",len(ede_positions))

In [ ]:
if build_again:
    build_uv_cache(lcdm_file=lcdm_file,ede_file=ede_file,cache_file=cache_file)
saved=np.load(cache_file)

print("cache arrays:",saved.files)
print("A1 galaxies:",len(saved["a1_positions"]))
print("C1 galaxies:",len(saved["c1_positions"]))
print("EDE galaxies:",len(saved["ede_positions"]))

In [ ]:
project_dir=Path(".")
position_file=project_dir/"cache/uv_samples_all_limits.npz"
saved=np.load(position_file)
a1_positions=saved["a1_positions"]
a1_muv=saved["a1_muv"]
c1_positions=saved["c1_positions"]
c1_muv=saved["c1_muv"]
ede_positions=saved["ede_positions"]
ede_muv=saved["ede_muv"]
field_lengths=saved["field_lengths"]
box_size=float(saved["box_size"])
muv_limits=saved["muv_limits"]

print("MUV limits:",muv_limits)
print("A1 parent sample:",len(a1_positions))
print("C1 parent sample:",len(c1_positions))
print("EDE parent sample:",len(ede_positions))
print("original field lengths [cMpc]:",field_lengths)

In [ ]:
#one map=16 original survey fields
maps_across_x=4
maps_across_y=4
map_lengths=field_lengths*np.array([maps_across_x,maps_across_y,1]) #just multiply
n_maps_side=np.floor(box_size/map_lengths).astype(int)
pixels_x=32
pixels_y=16
number_of_maps=int(np.prod(n_maps_side))

print("map lengths [cMpc]:",map_lengths)
print("maps along x,y,z:",n_maps_side)
print("number of maps:",number_of_maps)

In [ ]:
def make_maps(positions): #claude help with the logic of the function and how it works
    #need to make images because CNN reads images
    #galaxy positions-->divide simulation into regions-->divide region into pixels-->count galaxies in each pixel-->spatial maps
    maps=np.zeros((number_of_maps,pixels_y,pixels_x),dtype=np.float32)
    covered_x=n_maps_side[0]*map_lengths[0]
    covered_y=n_maps_side[1]*map_lengths[1]
    covered_z=n_maps_side[2]*map_lengths[2]
    for position in positions:
        x=position[0]
        y=position[1]
        z=position[2]
        # skip galaxies outside the covered map region
        if x>=covered_x or y>=covered_y or z>=covered_z:
            continue
        map_x=int(x/map_lengths[0]) # find which map contains the galaxy
        map_y=int(y/map_lengths[1])
        map_number=map_x*n_maps_side[1]+map_y # turn the 2D map location into one map number
        x_inside_map=x-map_x*map_lengths[0] # find the galaxy's position inside its map
        y_inside_map=y-map_y*map_lengths[1]
        pixel_x=int(x_inside_map/map_lengths[0]*pixels_x) #position --> a pixel
        pixel_y=int(y_inside_map/map_lengths[1]*pixels_y)

        # Add one galaxy to this pixel
        maps[map_number,pixel_y,pixel_x]+=1

    return maps

muv_limit=-19.5
# select galaxies at this magnitude limit
a1_selected=a1_positions[a1_muv<muv_limit]
c1_selected=c1_positions[c1_muv<muv_limit]
ede_selected=ede_positions[ede_muv<muv_limit]

print("Before matching:")
print("A1:",len(a1_selected))
print("C1:",len(c1_selected))
print("EDE:",len(ede_selected))

#matching the number of galaxies between models by randomly reducing every sample to the size of the smallest
# Find the smallest sample
target_number=min(len(a1_selected),len(c1_selected),len(ede_selected))
rng=np.random.default_rng(7)

# reduce A1 if necessary
if len(a1_selected)>target_number:
    keep=rng.choice(len(a1_selected),size=target_number,replace=False)
    a1_selected=a1_selected[keep]

# reduce C1 if necessary
if len(c1_selected)>target_number:
    keep=rng.choice(len(c1_selected),size=target_number,replace=False)
    c1_selected=c1_selected[keep]

# reduce EDE if necessary
if len(ede_selected)>target_number:
    keep=rng.choice(len(ede_selected),size=target_number,replace=False)
    ede_selected=ede_selected[keep]

print("After matching:",target_number,"galaxies per model")

# now make the maps
a1_maps=make_maps(a1_selected)
c1_maps=make_maps(c1_selected)
ede_maps=make_maps(ede_selected)
class_names=["A1","C1","EDE"]
all_maps=[a1_maps,c1_maps,ede_maps]

In [ ]:
#3 example maps!
map_number=0
maps_to_plot=[a1_maps[map_number],c1_maps[map_number],ede_maps[map_number]]
maximum_count=max(image.max() for image in maps_to_plot)
fig,axes=plt.subplots(1,3,figsize=(14,3),sharex=True,sharey=True,layout="constrained")
fig.patch.set_facecolor("white")

for class_number in range(3):
    image=axes[class_number].imshow(maps_to_plot[class_number],origin="lower",cmap="magma",vmin=0,vmax=maximum_count)
    axes[class_number].set_title(class_names[class_number])
    axes[class_number].set_xlabel("x pixel")
    axes[class_number].set_ylabel("y pixel")

fig.colorbar(image,ax=axes,label="number of galaxies",shrink=0.85)
fig.suptitle("matched spatial maps",fontsize=18)
plt.show()

In [ ]:
#calculate cosmic variance from summing all its pixels to get total galaxy count N, and we get sigma_cv from subtracting Poisson from the total variance. This is just a comparison
sigma_cv_values={}
for class_name,maps in zip(class_names,all_maps):
    counts=maps.sum(axis=(1,2))
    mean_count=counts.mean()
    total_variance=counts.var(ddof=1)
    sigma_cv=np.sqrt(max((total_variance-mean_count)/mean_count**2,0))
    sigma_cv_values[class_name]=sigma_cv
    print(class_name,"mean:",round(float(mean_count),3),"sigma_cv:",round(float(sigma_cv),3))

In [ ]:
#multi class CNN
class_names=["A1","C1","EDE"]
all_maps=[a1_maps,c1_maps,ede_maps]
number_of_classes=len(class_names)

print("classes:",class_names)
print("number of classes:",number_of_classes)

## Preparing dataset for training/testing/validation

So the map grid has ten x-columns, since the shape is (10,20,1). And we keep whole columns together so nearby regions don't get mixed up (this suggestion was taken from claude)

- 7 x-columns are used for training
- 1 is used for validation
- 2 are used for testing


In [ ]:
#we start with a1_maps, c1_maps and ede_maps storing all maps as a list of 200 maps
rng=np.random.default_rng(7)
x_columns=rng.permutation(n_maps_side[0])
train_columns=x_columns[:7]
validation_columns=x_columns[7:8]
test_columns=x_columns[8:]
train_maps=[]
validation_maps=[]
test_maps=[]

#geometry help from claude on how to construct the for loop
for x_column in train_columns: #loops over x,y locations into a map index because there are 10 x-columns and 20 y positions in each column
    for y_column in range(n_maps_side[1]):
        map_number=x_column*n_maps_side[1]+y_column
        train_maps.append(map_number) #adds the map indeices into the list

for x_column in validation_columns:
    for y_column in range(n_maps_side[1]):
        map_number=x_column*n_maps_side[1]+y_column
        validation_maps.append(map_number)

for x_column in test_columns:
    for y_column in range(n_maps_side[1]):
        map_number=x_column*n_maps_side[1]+y_column
        test_maps.append(map_number)

train_maps=np.array(train_maps)
validation_maps=np.array(validation_maps)
test_maps=np.array(test_maps)

print("training columns:",train_columns)
print("validation columns:",validation_columns)
print("test columns:",test_columns)
print("training maps per class:",len(train_maps))
print("validation maps per class:",len(validation_maps))
print("test maps per class:",len(test_maps))

In [ ]:
#label 0 is A1, label 1 is C1, and label 2 is EDE
raw_sets=[]
label_sets=[]

for map_ids in [train_maps,validation_maps,test_maps]:
    X=np.concatenate([a1_maps[map_ids],c1_maps[map_ids],ede_maps[map_ids]])
    y=np.concatenate([np.zeros(len(map_ids),dtype=np.int64),np.ones(len(map_ids),dtype=np.int64),np.full(len(map_ids),2,dtype=np.int64)]) #just makes labels of 0,1,2 for each class arrays
    raw_sets.append(X)
    label_sets.append(y)

X_train_raw,X_validation_raw,X_test_raw=raw_sets
y_train,y_validation,y_test=label_sets

print("training data:",X_train_raw.shape)
print("validation data:",X_validation_raw.shape)
print("test data:",X_test_raw.shape)
print("training labels:",y_train.shape)

In [ ]:
processed_sets=[]

for X in [X_train_raw,X_validation_raw,X_test_raw]:
    log_maps=np.log1p(X)
    processed_maps=log_maps.reshape(len(X),1,pixels_y,pixels_x).astype(np.float32)
    processed_sets.append(processed_maps)

X_train,X_validation,X_test=processed_sets

training_mean=X_train.mean()
training_std=X_train.std()

X_train=(X_train-training_mean)/training_std #normalize
X_validation=(X_validation-training_mean)/training_std
X_test=(X_test-training_mean)/training_std

print("CNN input shape:",X_train.shape)

In [ ]:
#build the model-> 2 layers, 2 ReLU, 2 max pool
class MapCNN(nn.Module):
    def __init__(self,number_of_classes):
        super().__init__()
        self.conv1=nn.Conv2d(1,12,kernel_size=3,padding=1)
        self.conv2=nn.Conv2d(12,24,kernel_size=3,padding=1)
        self.pool=nn.MaxPool2d(2)
        self.average_pool=nn.AdaptiveAvgPool2d(1)
        self.linear=nn.Linear(24,number_of_classes)

    def forward(self,x): #input (batch,1,16,32)
        x=F.relu(self.conv1(x))  #(batch,12,16,32)
        x=self.pool(x) #(batch,12,8,16)
        x=F.relu(self.conv2(x)) #(batch,24,8,16)
        x=self.pool(x) #(batch,24,4,8)
        x=self.average_pool(x) #(batch,24,1,1)
        x=x.flatten(start_dim=1) #(batch,24)
        x=self.linear(x) #(batch,3)
        return x

In [ ]:
def train_model(X_train,y_train,X_validation,y_validation,seed=7,epochs=30,number_of_classes=3):
    torch.manual_seed(seed)
    train_data=TensorDataset(torch.tensor(X_train,dtype=torch.float32),torch.tensor(y_train,dtype=torch.long))
    train_loader=DataLoader(train_data,batch_size=32,shuffle=True)
    validation_tensor=torch.tensor(X_validation,dtype=torch.float32).to(device)
    model=MapCNN(number_of_classes).to(device)
    optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
    loss_function=nn.CrossEntropyLoss()
    best_accuracy=-1
    best_state=None

    for epoch in range(epochs):
        model.train()
        for X_batch,y_batch in train_loader:
            X_batch=X_batch.to(device)
            y_batch=y_batch.to(device)
            optimizer.zero_grad()
            output=model(X_batch)
            loss=loss_function(output,y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        
        with torch.no_grad():
            validation_output=model(validation_tensor)
            validation_predictions=torch.argmax(validation_output,dim=1).cpu().numpy()

        validation_accuracy=np.mean(validation_predictions==y_validation) #claude help with this section as to why use validation set and logic
        if validation_accuracy>best_accuracy: #remembers the weights from the best validation model
            best_accuracy=validation_accuracy
            best_state=copy.deepcopy(model.state_dict())
        print("epoch",epoch+1,"validation accuracy",round(float(validation_accuracy),3))
    model.load_state_dict(best_state) #restored to best state
    print("best validation accuracy:",round(float(best_accuracy),3))
    return model

In [ ]:
best_model=train_model(X_train,y_train,X_validation,y_validation,number_of_classes=3)
best_model.eval()

with torch.no_grad():
    test_output=best_model(torch.tensor(X_test,dtype=torch.float32).to(device))
    predictions=torch.argmax(test_output,dim=1).cpu().numpy()

full_accuracy=np.mean(predictions==y_test)
print("full-map accuracy:",round(float(full_accuracy),3))

In [ ]:
#confusion matrix
confusion=np.zeros((3,3),dtype=int)
plt.style.use("default")
for prediction,true_label in zip(predictions,y_test):
    confusion[true_label,prediction]+=1

fig,ax=plt.subplots(figsize=(6,5),facecolor="white",layout="constrained")
image=ax.imshow(confusion,cmap="Blues",vmin=0,vmax=confusion.max(),interpolation="nearest")

colorbar=fig.colorbar(image,ax=ax,shrink=0.85,pad=0.04)
colorbar.set_label("number of maps")

ax.set_xticks([0,1,2])
ax.set_yticks([0,1,2])
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title("confusion matrix",fontsize=14,pad=12)

for true_class in range(3):
    row_total=confusion[true_class].sum()
    for predicted_class in range(3):
        count=confusion[true_class,predicted_class]
        text_color="white" if count>confusion.max()/2 else "black"
        ax.text(predicted_class,true_class,str(count), ha="center",va="center",color=text_color,fontsize=11)

ax.set_xticks(np.arange(-0.5,3,1),minor=True)
ax.set_yticks(np.arange(-0.5,3,1),minor=True)
ax.grid(which="minor",color="black",linewidth=1)
ax.tick_params(which="minor",bottom=False,left=False)
ax.tick_params(top=False,right=False)
ax.set_aspect("equal")
plt.show()

In [ ]:
seeds=[1,2,3,4,5]
seed_accuracies=[]
seed_confusions=[]

test_tensor=torch.tensor(X_test,dtype=torch.float32).to(device)

for seed in seeds:
    sweep_model=train_model(X_train,y_train,X_validation,y_validation,seed=seed,number_of_classes=3)
    sweep_model.eval()

    with torch.no_grad():
        sweep_output=sweep_model(test_tensor)
        sweep_predictions=torch.argmax(sweep_output,dim=1).cpu().numpy() #claude help with debugging here on how to vary seeds as a parameter

    sweep_accuracy=np.mean(sweep_predictions==y_test)
    seed_accuracies.append(sweep_accuracy)

    sweep_confusion=np.zeros((3,3),dtype=int)

    for prediction,true_label in zip(sweep_predictions,y_test):
        sweep_confusion[true_label,prediction]+=1

    seed_confusions.append(sweep_confusion)

    print("seed",seed)
    print("accuracy:",round(float(sweep_accuracy),3))
    print("number predicted as [A1,C1,EDE]:",sweep_confusion.sum(axis=0))

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(12,8),facecolor="white",layout="constrained")
axes=axes.flatten()

for seed_number in range(len(seeds)):
    ax=axes[seed_number]
    confusion=seed_confusions[seed_number]
    accuracy=seed_accuracies[seed_number]

    image=ax.imshow(confusion,cmap="Blues",vmin=0,vmax=40,interpolation="nearest")
    ax.set_xticks([0,1,2])
    ax.set_yticks([0,1,2])
    ax.set_xticklabels(["A1","C1","EDE"])
    ax.set_yticklabels(["A1","C1","EDE"])
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title("seed "+str(seeds[seed_number])+"\naccuracy="+str(round(float(accuracy),3)))

    for true_class in range(3):
        for predicted_class in range(3):
            count=confusion[true_class,predicted_class]
            text_color="black"
            ax.text(predicted_class,true_class,str(count),ha="center",va="center",color=text_color)

    ax.set_xticks(np.arange(-0.5,3,1),minor=True)
    ax.set_yticks(np.arange(-0.5,3,1),minor=True)
    ax.grid(which="minor",color="black",linewidth=1)
    ax.tick_params(which="minor",bottom=False,left=False)

axes[5].axis("off")
fig.colorbar(image,ax=axes[:5].tolist(),label="number of maps",shrink=0.8)
fig.suptitle("Three-class confusion matrices across seeds",fontsize=15)
plt.show()

now we use shuffle maps as control--> so it contains the exact same pixel values as the original but those values are moved to random locations. If the full maps perform better, their arrangement contains useful information.

In [ ]:
rng=np.random.default_rng(100)
#make copies so the original maps are unchanged
shuffled_train_raw=X_train_raw.copy()
shuffled_validation_raw=X_validation_raw.copy()
shuffled_test_raw=X_test_raw.copy()

#shuffle the pixels inside every map, claude help with this one
for maps in [shuffled_train_raw,shuffled_validation_raw,shuffled_test_raw]:
    for map_number in range(len(maps)):
        pixels=maps[map_number].flatten()
        rng.shuffle(pixels)
        maps[map_number]=pixels.reshape(pixels_y,pixels_x)

#apply the same processing used for the original maps
shuffled_train=np.log1p(shuffled_train_raw)
shuffled_validation=np.log1p(shuffled_validation_raw)
shuffled_test=np.log1p(shuffled_test_raw)

shuffled_train=shuffled_train.reshape(len(shuffled_train),1,pixels_y,pixels_x).astype(np.float32)
shuffled_validation=shuffled_validation.reshape(len(shuffled_validation),1,pixels_y,pixels_x).astype(np.float32)
shuffled_test=shuffled_test.reshape(len(shuffled_test),1,pixels_y,pixels_x).astype(np.float32)

#normalize using only the shuffled training maps
shuffled_mean=shuffled_train.mean()
shuffled_std=shuffled_train.std()

shuffled_train=(shuffled_train-shuffled_mean)/shuffled_std
shuffled_validation=(shuffled_validation-shuffled_mean)/shuffled_std
shuffled_test=(shuffled_test-shuffled_mean)/shuffled_std

In [ ]:
shuffled_model=train_model(shuffled_train,y_train,shuffled_validation,y_validation,number_of_classes=3)
shuffled_model.eval()

with torch.no_grad():
    shuffled_output=shuffled_model(torch.tensor(shuffled_test,dtype=torch.float32).to(device))
    shuffled_predictions=torch.argmax(shuffled_output,dim=1).cpu().numpy()

shuffled_accuracy=np.mean(shuffled_predictions==y_test)
print("shuffled map accuracy:",round(float(shuffled_accuracy),3))

another control--> this is what we currently do!
each field, or "map" is represented by only one number, which is the total number of galaxies. It sees no spatial locations and we would compute the cosmic variance from only the number of galaxies alone.

In [ ]:
#this one only gets a number, and we use logistic regression since we're using numbers
train_counts=X_train_raw.sum(axis=(1,2)).reshape(-1,1)
test_counts=X_test_raw.sum(axis=(1,2)).reshape(-1,1)
count_model=LogisticRegression()
count_model.fit(train_counts,y_train)
count_accuracy=count_model.score(test_counts,y_test)
print("count-only accuracy:",round(float(count_accuracy),3))

In [ ]:
method_names=["count","shuffled","full"]
accuracies=[count_accuracy,shuffled_accuracy,full_accuracy]
chance_accuracy=1/number_of_classes

fig,ax=plt.subplots(figsize=(7,4.5),facecolor="white")
bars=ax.bar(method_names,accuracies,color=["gray","tab:orange","tab:blue"])
ax.axhline(chance_accuracy,color="black",linestyle="--",label="random")
ax.set_ylim(0,1)
ax.set_ylabel("test accuracy")
ax.set_title("A1, C1, and EDE Classification")
ax.legend()

for bar,accuracy in zip(bars,accuracies):
    ax.text(bar.get_x()+bar.get_width()/2,accuracy,str(round(float(accuracy),3)),ha="center")

plt.tight_layout()
plt.show()

Next, probe the models two at a time with binary classifiers.

In [ ]:
#basically copy and paste the same pipeline but now for 2 classes insstead of 3
def run_binary_comparison(name_a,maps_a,name_b,maps_b,epochs=30):
    #label 0 is the first model, label 1 is the second model
    raw_sets=[]
    label_sets=[]

    for map_ids in [train_maps,validation_maps,test_maps]:
        X=np.concatenate([maps_a[map_ids],maps_b[map_ids]])
        y=np.concatenate([np.zeros(len(map_ids),dtype=np.int64),np.ones(len(map_ids),dtype=np.int64)])
        raw_sets.append(X)
        label_sets.append(y)

    X_train_raw,X_validation_raw,X_test_raw=raw_sets
    y_train,y_validation,y_test=label_sets

    processed_sets=[]

    for X in [X_train_raw,X_validation_raw,X_test_raw]:
        log_maps=np.log1p(X)
        processed_maps=log_maps.reshape(len(X),1,pixels_y,pixels_x).astype(np.float32)
        processed_sets.append(processed_maps)

    X_train,X_validation,X_test=processed_sets

    training_mean=X_train.mean()
    training_std=X_train.std()

    X_train=(X_train-training_mean)/training_std #normalize
    X_validation=(X_validation-training_mean)/training_std
    X_test=(X_test-training_mean)/training_std

    best_model=train_model(X_train,y_train,X_validation,y_validation,epochs=epochs,number_of_classes=2)
    best_model.eval()

    with torch.no_grad():
        test_output=best_model(torch.tensor(X_test,dtype=torch.float32).to(device))
        predictions=torch.argmax(test_output,dim=1).cpu().numpy()

    full_accuracy=np.mean(predictions==y_test)

    rng=np.random.default_rng(100)
    #make copies so the original maps are unchanged
    shuffled_train_raw=X_train_raw.copy()
    shuffled_validation_raw=X_validation_raw.copy()
    shuffled_test_raw=X_test_raw.copy()

    #shuffle the pixels inside every map
    for maps in [shuffled_train_raw,shuffled_validation_raw,shuffled_test_raw]:
        for map_number in range(len(maps)):
            pixels=maps[map_number].flatten()
            rng.shuffle(pixels)
            maps[map_number]=pixels.reshape(pixels_y,pixels_x)

    #apply the same processing used for the original maps
    shuffled_train=np.log1p(shuffled_train_raw)
    shuffled_validation=np.log1p(shuffled_validation_raw)
    shuffled_test=np.log1p(shuffled_test_raw)

    shuffled_train=shuffled_train.reshape(len(shuffled_train),1,pixels_y,pixels_x).astype(np.float32)
    shuffled_validation=shuffled_validation.reshape(len(shuffled_validation),1,pixels_y,pixels_x).astype(np.float32)
    shuffled_test=shuffled_test.reshape(len(shuffled_test),1,pixels_y,pixels_x).astype(np.float32)

    #normalize using only the shuffled training maps
    shuffled_mean=shuffled_train.mean()
    shuffled_std=shuffled_train.std()

    shuffled_train=(shuffled_train-shuffled_mean)/shuffled_std
    shuffled_validation=(shuffled_validation-shuffled_mean)/shuffled_std
    shuffled_test=(shuffled_test-shuffled_mean)/shuffled_std

    shuffled_model=train_model(shuffled_train,y_train,shuffled_validation,y_validation,epochs=epochs,number_of_classes=2)
    shuffled_model.eval()

    with torch.no_grad():
        shuffled_output=shuffled_model(torch.tensor(shuffled_test,dtype=torch.float32).to(device))
        shuffled_predictions=torch.argmax(shuffled_output,dim=1).cpu().numpy()

    shuffled_accuracy=np.mean(shuffled_predictions==y_test)

    #this one only gets a number, and we use logistic regression since we're using numbers
    train_counts=X_train_raw.sum(axis=(1,2)).reshape(-1,1)
    test_counts=X_test_raw.sum(axis=(1,2)).reshape(-1,1)
    count_model=LogisticRegression()
    count_model.fit(train_counts,y_train)
    count_accuracy=count_model.score(test_counts,y_test)

    print(f"\n{name_a} versus {name_b}")
    print("count-only accuracy:",round(float(count_accuracy),3))
    print("shuffled map accuracy:",round(float(shuffled_accuracy),3))
    print("full-map accuracy:",round(float(full_accuracy),3))

    return {"count":count_accuracy,"shuffled":shuffled_accuracy,"full":full_accuracy}

In [ ]:
#run each pair 
pair_results={}
pair_results["C1 versus EDE"]=run_binary_comparison("C1",c1_maps,"EDE",ede_maps)
pair_results["C1 versus A1"]=run_binary_comparison("C1",c1_maps,"A1",a1_maps)

In [ ]:
#grouped bar chart comparing both pairs across the three methods
plt.style.use("default")
method_names=["count","shuffled","full"]
pair_names=list(pair_results.keys())
chance_accuracy=0.5

fig,ax=plt.subplots(figsize=(8,4.5),facecolor="white")
bar_width=0.35
x_positions=np.arange(len(method_names))

for pair_number,pair_name in enumerate(pair_names):
    accuracies=[pair_results[pair_name][method] for method in method_names]
    offset=(pair_number-0.5)*bar_width
    bars=ax.bar(x_positions+offset,accuracies,bar_width,label=pair_name)
    for bar,accuracy in zip(bars,accuracies):
        ax.text(bar.get_x()+bar.get_width()/2,accuracy+0.02,str(round(float(accuracy),3)),ha="center",fontsize=9)

ax.axhline(chance_accuracy,color="black",linestyle="--",label="random")
ax.set_xticks(x_positions)
ax.set_xticklabels(method_names)
ax.set_ylim(0,1)
ax.set_ylabel("test accuracy")
ax.set_title("binary comparison")
ax.legend()
plt.tight_layout()
plt.show()

## Interpretations

Results are not included here because they use the Lumina EDE simulation, which is not public. This project is in progress; see the README for next steps.